# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIRˆ² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Display dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

We inspect the high-level structure to find record set `@id`s and their respective field `@id`s. All references below use the Croissant schema's `@id`.

In [ ]:
from pprint import pprint

# List all available record sets with their @id and description
print('Record Sets and their Fields:')
for rs in metadata.record_sets:
    print(f"- Record Set: {rs['@id']} (name: {rs.get('name', '')})")
    if 'fields' in rs:
        for field in rs['fields']:
            print(f"    - Field: {field['@id']} (name: {field.get('name', '')}, dtype: {field.get('dataType', '')})")

If the dataset contains multiple record sets, select the `@id` corresponding to your analysis of interest below. If unsure, use the first listed.

In [ ]:
# Show several example records for the first record set
available_record_sets = [rs['@id'] for rs in metadata.record_sets]
record_set_id = available_record_sets[0]

# Display a sample of records from the first record set
pprint([r for r,_ in zip(dataset.records(record_set=record_set_id), range(3))])

## 3. Data Extraction
Load data from all record sets into pandas DataFrames for analysis. Use the record set and field `@id`s found above.

In [ ]:
# Extract all record sets to DataFrames by @id
dataframes = {}
for rs in metadata.record_sets:
    rs_id = rs['@id']
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

print(f"DataFrame columns for record set '{record_set_id}':\n", dataframes[record_set_id].columns.tolist())
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping. 

Select a numeric field (`@id`) for analysis. If unsure, pick the field with type `Float` or `Integer` in the record set above.

In [ ]:
# Example: automatically select a numeric field if present
target_record_set = metadata.record_sets[0]
num_field_id = None
for field in target_record_set['fields']:
    if field.get('dataType', '').lower() in {'float', 'integer', 'number'}:
        num_field_id = field['@id']
        break

if num_field_id:
    numeric_field = num_field_id
    print(f"Selected numeric field: {numeric_field}")

    df = dataframes[record_set_id]
    # Ensure the column is numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    threshold = df[numeric_field].mean()  # Use mean as an arbitrary threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to find a string/nominal field for grouping
    group_field_id = None
    for field in target_record_set['fields']:
        if field.get('dataType', '').lower() in {'string', 'text'}:
            group_field_id = field['@id']
            break
    if group_field_id and group_field_id in df.columns:
        group_field = group_field_id
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields.

We will plot the distribution of the selected numeric field, and if grouping field exists, plot its means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if num_field_id:
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    if 'group_field' in locals():
        plt.figure(figsize=(10,2))
        sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field])
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.title(f"Mean {numeric_field} per {group_field}")
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
In this notebook, we loaded the mass spectrometry extracellular vesicle dataset using `mlcroissant`, explored its metadata and structure by `@id`, extracted the primary record set to a DataFrame, identified and processed a numeric field for basic analysis, and visualized key attributes. For in-depth analysis, users may further explore additional record sets and fields, adjusting filtering and transformations as appropriate for their research purposes.